# MANIFOLD V2 — MAMO (Multi-Agent Multi-Objective)

**MANIFOLD**: Multi-Agent Non-stationary Framework for Ontogenetic Learning and Dynamic valuation

## Objectives
* Extend the static grid with **congestion**, **risk**, and **cost** as independent objective axes
* Introduce multiple *vector agents*, each with its own physics parameters (`riskMultiplier`, `maxRisk`)
* Show that the same cell has different utility depending on agent physics — value is **subjective**
* Measure efficiency under competition: performance is no longer shortest path but weighted cost under congestion

## Inputs
* Static cell values from V1 (recomputed inline)
* Agent physics sampled from predefined ranges

## Outputs
* Congestion maps over time
* Per-agent utility surfaces
* Efficiency-under-competition trajectories
* Pareto frontier visualisation (cost vs risk)

## Additional Comments
* MAMO treats the value grid as a backdrop and adds *physics diversity* — the first step toward the emergent V3 system.

---

## Setup

In [ ]:
import os
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

WINNING_ROUTES = [
    [0,1,2],[3,4,5],[6,7,8],
    [0,3,6],[1,4,7],[2,5,8],
    [0,4,8],[2,4,6],
]
route_count = np.zeros(9, dtype=int)
for route in WINNING_ROUTES:
    for c in route:
        route_count[c] += 1
cell_value = route_count / len(WINNING_ROUTES)
GRID_ROWS, GRID_COLS = 3, 3

def cell_to_rc(cell): return divmod(cell, GRID_COLS)
def rc_to_cell(r, c): return r * GRID_COLS + c
def neighbours(cell):
    r, c = cell_to_rc(cell)
    return [rc_to_cell(nr, nc)
            for nr, nc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)]
            if 0 <= nr < GRID_ROWS and 0 <= nc < GRID_COLS]

print('Setup complete. Cell values:', cell_value)

---
## Section 1 — Agent physics and multi-objective cost function

Each vector agent carries two physics parameters:
- **`riskMultiplier`** (ρ): how much the agent penalises risk relative to travel cost
- **`maxRisk`** (μ): the risk threshold above which a cell is *impassable* for this agent

The multi-objective cost to enter cell $s'$ is:
$$C(s') = \underbrace{(1 - V(s'))}_{\text{travel cost}} + \underbrace{\rho \cdot R(s')}_{\text{risk penalty}} + \underbrace{\kappa \cdot N(s')}_{\text{congestion penalty}}$$

where $R(s')$ is cell risk, $N(s')$ is the number of other agents currently in $s'$, and $\kappa$ is the global congestion weight.

In [ ]:
KAPPA = 0.15   # congestion weight

AGENT_TEMPLATES = [
    {'name': 'Tank',    'riskMultiplier': 0.2,  'maxRisk': 8.5, 'color': '#e74c3c'},
    {'name': 'Scout',   'riskMultiplier': 1.8,  'maxRisk': 3.5, 'color': '#3498db'},
    {'name': 'Balanced','riskMultiplier': 0.9,  'maxRisk': 6.0, 'color': '#2ecc71'},
    {'name': 'Berserker','riskMultiplier': 0.05, 'maxRisk': 9.5, 'color': '#f39c12'},
    {'name': 'Coward',  'riskMultiplier': 2.5,  'maxRisk': 2.0, 'color': '#9b59b6'},
]

# Static risk map: spikes at certain cells to show differential routing
base_risk = np.array([1.0, 2.0, 1.0,
                      3.5, 6.0, 3.5,
                      1.0, 2.0, 1.0])

def multi_obj_cost(dst, agent, congestion_map):
    r = base_risk[dst]
    if r > agent['maxRisk']:
        return float('inf')
    travel = 1.0 - cell_value[dst]
    risk_pen = agent['riskMultiplier'] * r
    congestion_pen = KAPPA * congestion_map[dst]
    return travel + risk_pen + congestion_pen

def manhattan(a, b):
    r1, c1 = cell_to_rc(a)
    r2, c2 = cell_to_rc(b)
    return abs(r1 - r2) + abs(c1 - c2)

def astar_mamo(start, goal, agent, congestion_map):
    open_heap = [(manhattan(start, goal), 0.0, start, [start])]
    visited = {}
    while open_heap:
        f, g, current, path = heapq.heappop(open_heap)
        if current in visited:
            continue
        visited[current] = g
        if current == goal:
            return path, g
        for nb in neighbours(current):
            if nb not in visited:
                step = multi_obj_cost(nb, agent, congestion_map)
                if step < float('inf'):
                    ng = g + step
                    heapq.heappush(open_heap,
                                   (ng + manhattan(nb, goal), ng, nb, path + [nb]))
    return None, float('inf')

# Baseline: each agent routes from cell 0 → 8 on an empty grid
empty_congestion = np.zeros(9)
print('Agent routing on empty grid (0→8):')
for agent in AGENT_TEMPLATES:
    path, cost = astar_mamo(0, 8, agent, empty_congestion)
    print(f"  {agent['name']:10s}: path={path}  cost={cost:.3f}")

---
## Section 2 — Per-agent utility surfaces

For each agent type we compute the effective *utility* of every cell: $U_i(s) = V(s) - \rho_i \cdot R(s)$. This shows that the same spatial geometry yields radically different utility landscapes depending on physics.

In [ ]:
cmap_utility = LinearSegmentedColormap.from_list(
    'utility', ['#1a1a2e', '#16213e', '#0f3460', '#533483', '#e94560'], N=256)

fig, axes = plt.subplots(1, len(AGENT_TEMPLATES), figsize=(20, 4))
fig.patch.set_facecolor('#111111')

for ax, agent in zip(axes, AGENT_TEMPLATES):
    utility = cell_value - agent['riskMultiplier'] * base_risk / 10.0
    # cells above maxRisk are dead zones
    dead = base_risk > agent['maxRisk']
    grid_u = utility.reshape(3, 3)
    grid_dead = dead.reshape(3, 3)

    im = ax.imshow(grid_u, cmap=cmap_utility)
    for r in range(3):
        for c in range(3):
            cell = rc_to_cell(r, c)
            txt = f'{utility[cell]:.2f}'
            if dead[cell]:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1,
                             color='#333', zorder=2))
                ax.text(c, r, 'X', ha='center', va='center',
                        fontsize=14, color='red', fontweight='bold', zorder=3)
            else:
                ax.text(c, r, txt, ha='center', va='center',
                        fontsize=11, color='white', fontweight='bold')
    ax.set_title(f"{agent['name']}\nρ={agent['riskMultiplier']}, μ={agent['maxRisk']}",
                 color='white', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor('#0d0d0d')

plt.suptitle('Per-Agent Utility Surfaces — Same Grid, Different Physics',
             fontsize=13, color='white', y=1.02)
plt.tight_layout()
plt.savefig('v2_utility_surfaces.png', dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()
print('Saved v2_utility_surfaces.png')

**Key insight**: The center cell (highest V1 value) is a **dead zone** for the Coward and a marginal choice for the Scout. Value is not intrinsic to geometry — it is a function of agent physics.

---
## Section 3 — Multi-agent simulation with congestion

We run N=20 agents simultaneously, all routing from cell 0 to cell 8. After each agent commits to a path, it increments the congestion counter for each cell it occupies. Later agents see increased costs in crowded cells.

In [ ]:
np.random.seed(0)
N_AGENTS = 20

# Spawn agents by randomly sampling physics from the MAMO ranges
def spawn_agent(idx):
    rm = np.random.uniform(0.1, 2.5)
    mr = np.random.uniform(2.0, 9.5)
    # Assign a template label for plotting
    if rm < 0.4:
        label = 'Tank'
    elif rm > 1.5:
        label = 'Scout' if mr < 5 else 'Coward'
    else:
        label = 'Balanced'
    return {'id': idx, 'name': label, 'riskMultiplier': rm, 'maxRisk': mr}

agents = [spawn_agent(i) for i in range(N_AGENTS)]

congestion = np.zeros(9)
records = []

for agent in agents:
    path, cost = astar_mamo(0, 8, agent, congestion)
    if path is None:
        path, cost = [0, 8], float('inf')  # fallback
    for cell in path:
        congestion[cell] += 1
    records.append({
        'id': agent['id'],
        'type': agent['name'],
        'riskMult': round(agent['riskMultiplier'], 3),
        'maxRisk': round(agent['maxRisk'], 3),
        'path': path,
        'path_len': len(path),
        'cost': round(cost, 4),
        'uses_center': 4 in (path or []),
    })

df = pd.DataFrame(records)
print(f'Simulation complete. {len(df)} agents routed.')
print(f'Center cell (4) usage: {df["uses_center"].sum()} / {len(df)} agents')
print('\nFinal congestion map:')
print(congestion.reshape(3, 3).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#111111')

# Congestion heatmap
ax = axes[0]
cong_grid = congestion.reshape(3, 3)
cmap_cong = LinearSegmentedColormap.from_list('cong', ['#0d0d0d', '#533483', '#e94560'], N=256)
im = ax.imshow(cong_grid, cmap=cmap_cong)
for r in range(3):
    for c in range(3):
        ax.text(c, r, str(int(cong_grid[r, c])), ha='center', va='center',
                fontsize=14, color='white', fontweight='bold')
plt.colorbar(im, ax=ax, label='Agent visits')
ax.set_title('Final Congestion Map\n(20 agents, 0→8)', color='white', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
ax.set_facecolor('#0d0d0d')

# Cost distribution by agent type
ax2 = axes[1]
type_colors_map = {'Tank': '#e74c3c', 'Scout': '#3498db', 'Balanced': '#2ecc71',
                   'Coward': '#9b59b6', 'Berserker': '#f39c12'}
for atype in df['type'].unique():
    subset = df[df['type'] == atype]
    ax2.scatter(subset['riskMult'], subset['cost'],
                color=type_colors_map.get(atype, 'white'),
                label=atype, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
ax2.set_xlabel('riskMultiplier (ρ)', color='white')
ax2.set_ylabel('Path cost', color='white')
ax2.set_title('Cost vs Risk Multiplier', color='white', fontsize=11)
ax2.set_facecolor('#0d0d0d')
ax2.tick_params(colors='white')
ax2.legend(fontsize=9)
for spine in ax2.spines.values(): spine.set_edgecolor('#444')

# Center usage rate
ax3 = axes[2]
center_by_type = df.groupby('type')['uses_center'].mean() * 100
bars = ax3.bar(center_by_type.index,
               center_by_type.values,
               color=[type_colors_map.get(t, 'white') for t in center_by_type.index],
               edgecolor='white', linewidth=0.8)
for bar in bars:
    h = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2, h + 1,
             f'{h:.0f}%', ha='center', va='bottom', color='white', fontsize=10)
ax3.set_ylabel('% agents using center cell', color='white')
ax3.set_title('Center-Cell Usage by Agent Type', color='white', fontsize=11)
ax3.set_facecolor('#0d0d0d')
ax3.tick_params(colors='white')
ax3.set_ylim(0, 115)
for spine in ax3.spines.values(): spine.set_edgecolor('#444')

plt.suptitle('MANIFOLD V2 — Multi-Agent Simulation Results', fontsize=14,
             color='white', y=1.02)
plt.tight_layout()
plt.savefig('v2_mamo_sim.png', dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()
print('Saved v2_mamo_sim.png')

---
## Section 4 — Pareto frontier: cost vs risk

In a multi-objective setting we cannot claim a single *best* path. Instead we find the Pareto frontier — paths that minimise cost without a dominating alternative on the risk axis.

In [ ]:
def path_risk(path):
    return sum(base_risk[c] for c in path) if path else float('inf')

df['path_risk'] = df['path'].apply(path_risk)

# Pareto filter: a point (cost, risk) is non-dominated if no other point
# has strictly lower cost AND strictly lower risk
def is_pareto(costs, risks):
    n = len(costs)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i != j and costs[j] <= costs[i] and risks[j] <= risks[i] and \
               (costs[j] < costs[i] or risks[j] < risks[i]):
                dominated[i] = True
                break
    return ~dominated

finite = df[df['cost'] < float('inf')].copy()
pareto_mask = is_pareto(finite['cost'].values, finite['path_risk'].values)
pareto_df = finite[pareto_mask].sort_values('cost')

print(f'Non-dominated solutions: {pareto_mask.sum()} / {len(finite)}')
print(pareto_df[['id', 'type', 'riskMult', 'cost', 'path_risk', 'path']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#0d0d0d')

# All points
for atype in finite['type'].unique():
    sub = finite[finite['type'] == atype]
    ax.scatter(sub['cost'], sub['path_risk'],
               color=type_colors_map.get(atype, 'white'),
               label=atype, s=60, alpha=0.6, edgecolors='none')

# Pareto front
ax.scatter(pareto_df['cost'], pareto_df['path_risk'],
           color='white', s=120, zorder=5, marker='*', label='Pareto front')
ax.step(pareto_df['cost'], pareto_df['path_risk'],
        where='post', color='white', linewidth=1.5, linestyle='--', alpha=0.7)

ax.set_xlabel('Total path cost', color='white')
ax.set_ylabel('Total path risk', color='white')
ax.set_title('Pareto Frontier — Cost vs Risk\n(20 agents, 0→8)', color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#444')
ax.legend(fontsize=9, framealpha=0.2)

plt.tight_layout()
plt.savefig('v2_pareto.png', dpi=150, bbox_inches='tight', facecolor='#111111')
plt.show()
print('Saved v2_pareto.png')

---
## Section 5 — Conclusions and Next Steps

### What MAMO proved
1. **Value is subjective** — the center cell is desirable for Tanks (low ρ, high μ) but passable-if-costly or even impassable for Scouts and Cowards.
2. **Congestion redistributes routes** — as the center fills up, later agents find cheaper paths around it, emergently balancing load.
3. **Pareto thinking is necessary** — in multi-objective settings there is no single best path; the Pareto frontier is the true performance surface.

### The remaining limitation
Physics are still *pre-assigned* at spawn — agents don't learn or adapt. The grid is still *static*. These constraints are lifted in MANIFOLD V3.

### Key numbers to carry forward
| Metric | Value |
|---|---|
| Agents using center cell (empty grid) | Majority of low-risk agents |
| Center congestion after 20 agents | Highest in grid |
| Pareto-optimal solutions | Minority — diversity is structurally necessary |
| riskMultiplier range | 0.1 – 2.5 |
| maxRisk range | 2.0 – 9.5 |

In [ ]:
print('MANIFOLD V2 — MAMO: complete.')
print('Outputs: v2_utility_surfaces.png, v2_mamo_sim.png, v2_pareto.png')